
# Huge Ensembles (HENS)

workflow พื้นฐานสำหรับการใช้งาน Huge Ensemble checkpoints หลายชุด

ตัวอย่างนี้แสดงวิธีโหลด Huge Ensemble checkpoints เพื่อรัน ensemble inference โดยมีจุดมุ่งหมายเพื่อสาธิตพื้นฐานของ multi-checkpoint workflow จากคอมโพเนนต์ของ Earth2Studio

สำหรับรายละเอียดเพิ่มเติมเกี่ยวกับ HENS โปรดดู:

- https://arxiv.org/abs/2408.03100
- https://github.com/ankurmahesh/earth2mip-fork

<div class="alert alert-danger"><h4>คำเตือน</h4><p>ขอแนะนำให้ผู้ใช้ตรวจสอบข้อจำกัดด้านใบอนุญาตของ checkpoint โมเดลนี้ก่อนใช้งาน</p></div>

สำหรับ HENS workflow แบบครบถ้วน เราแนะนำให้ดู recipe ของ HENS ซึ่งเป็นโซลูชันแบบ end-to-end สำหรับนำ HENS ไปใช้กับงานวิเคราะห์ขั้นปลาย เช่น การติดตามพายุหมุนเขตร้อน:

- https://github.com/NVIDIA/earth2studio/tree/main/recipes/hens

ในตัวอย่างนี้คุณจะได้เรียนรู้:

- วิธีโหลด HENS checkpoints ด้วย model package แบบกำหนดเอง
- วิธีโหลดวิธี perturbation ของ HENS
- วิธีสร้างลูป ensemble inference แบบง่าย
- วิธีพล็อตผลลัพธ์


In [ ]:
# /// script
# dependencies = [
#   "torch==2.9.1", # Match lock file to avoid torch-harmonics issue
#   "earth2studio[sfno] @ git+https://github.com/NVIDIA/earth2studio.git",
#   "cartopy",
# ]
# ///

## การเตรียมองค์ประกอบ
เริ่มต้นด้วยการนำเข้าโมดูลที่จำเป็นและตั้งค่าสภาพแวดล้อมสำหรับการทำงาน HENS มี checkpoint ให้ใช้งานอยู่บน [HuggingFace](https://huggingface.co/datasets/maheshankur10/hens/tree/main/earth2mip_prod_registry) ซึ่งเราจะใช้ในตัวอย่างนี้

แทนที่จะโหลด checkpoint เริ่มต้นจากงาน SFNO ต้นฉบับ เราจะสร้าง model package ที่ชี้ไปยัง HENS checkpoint เฉพาะที่ต้องการใช้งาน

ตัวอย่างนี้ยังต้องใช้องค์ประกอบต่อไปนี้:

- Prognostic Model พื้นฐาน: ใช้สถาปัตยกรรมโมเดล SFNO :py:class:`earth2studio.models.px.SFNO`
- Datasource: ดึงข้อมูลจาก GFS data API ผ่าน :py:class:`earth2studio.data.GFS`.
- Perturbation Method: HENS ใช้วิธี perturbation แบบใหม่ :py:class:`earth2studio.perturbation.HemisphericCentredBredVector`
- Seeding Method: ใช้วิธี Correlated Spherical Gaussian :py:class:`earth2studio.perturbation.CorrelatedSphericalGaussian`



In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()  # สิ่งที่ต้องทำ: สร้างฟังก์ชันการเตรียมตัวอย่างทั่วไป

from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.auto import Package
from earth2studio.models.px import SFNO
from earth2studio.perturbation import (
    CorrelatedSphericalGaussian,
    HemisphericCentredBredVector,
)
from earth2studio.run import ensemble

# ตั้งค่าแพ็คเกจสองโมเดลสำหรับแต่ละ checkpoint
# สังเกตการแก้ไขตำแหน่งแคชเพื่อหลีกเลี่ยงการเขียนทับ
model_package_1 = Package(
    "hf://datasets/maheshankur10/hens/earth2mip_prod_registry/sfno_linear_74chq_sc2_layers8_edim620_wstgl2-epoch70_seed102",
    cache_options={
        "cache_storage": Package.default_cache("hens_1"),
        "same_names": True,
    },
)

model_package_2 = Package(
    "hf://datasets/maheshankur10/hens/earth2mip_prod_registry/sfno_linear_74chq_sc2_layers8_edim620_wstgl2-epoch70_seed103",
    cache_options={
        "cache_storage": Package.default_cache("hens_2"),
        "same_names": True,
    },
)

# สร้างแหล่งข้อมูล
data = GFS()

## การรัน Workflow
ต่อไป เราจะดำเนินการ ensemble workflow สำหรับแต่ละรุ่น แต่วนซ้ำผ่าน checkpoint แต่ละตัว
โปรดทราบว่ายังไม่ได้โหลดโมเดลเหล่านี้ลงในหน่วยความจำ แต่จะเป็นเช่นนั้น
ทำทีละครั้งเพื่อลดขนาดหน่วยความจำของ inference บน GPU ให้เหลือน้อยที่สุด
ก่อนที่ ensemble workflow จะสามารถดำเนินการได้ จำเป็นต้องมีการตั้งค่าต่อไปนี้:

- เริ่มต้นโมเดล SFNO จาก checkpoint
- เริ่มต้นวิธีการก่อกวนด้วย Prognostic Model
- เตรียมใช้งานร้านค้า IO zarr สำหรับรุ่นนี้

หากมีการใช้ GPU หลายตัว ตัวหนึ่งสามารถขนาน inference โดยใช้ GPU ที่แตกต่างกันได้
checkpoints บนการ์ดแต่ละใบ



In [ ]:
import gc
from datetime import datetime, timedelta

import numpy as np
import torch

start_date = datetime(2024, 1, 1)
nsteps = 4
nensemble = 2

for i, package in enumerate([model_package_1, model_package_2]):
    # โหลดโมเดล SFNO จากแพ็คเกจ
    # HENS checkpoint ใช้อินพุตที่แตกต่างจาก SFNO เริ่มต้น (รวม d2m)
    # สามารถค้นหาสิ่งนี้ได้ใน config.json ฟังก์ชัน load_model ใน SFNO จะจัดการสิ่งนี้
    model = SFNO.load_model(package)

    # วิธีการก่อกวน
    # Here we will simplify the process that's in the original paper for conciseness
    noise_amplification = torch.zeros(model.input_coords()["variable"].shape[0])
    index_z500 = list(model.input_coords()["variable"]).index("z500")
    noise_amplification[index_z500] = 39.27  # z500 (0.35 * ทักษะ z500)
    noise_amplification = noise_amplification.reshape(1, 1, 1, -1, 1, 1)

    seed_perturbation = CorrelatedSphericalGaussian(noise_amplitude=noise_amplification)
    perturbation = HemisphericCentredBredVector(
        model, data, seed_perturbation, noise_amplitude=noise_amplification
    )

    # อ็อบเจ็กต์ IO
    io = ZarrBackend(
        file_name=f"outputs/11_hens_{i}.zarr",
        chunks={"ensemble": 1, "time": 1, "lead_time": 1},
        backend_kwargs={"overwrite": True},
    )

    io = ensemble(
        ["2024-01-01"],
        nsteps,
        nensemble,
        model,
        data,
        io,
        perturbation,
        batch_size=1,
        output_coords={"variable": np.array(["u10m", "v10m"])},
    )

    print(io.root.tree())
    # ทำการล้างข้อมูลด้วยตนเองเพื่อเพิ่ม VRAM
    del model
    del perturbation
    gc.collect()
    torch.cuda.empty_cache()

## การทำ Post-Processing
ผลลัพธ์ของ workflow คือร้านค้า zarr สองแห่งที่มีข้อมูล ensemble สำหรับ
checkpoint ตามลำดับที่ใช้
ตัวอย่างที่เหลือจะเน้นที่การประมวลผลขั้นพื้นฐานเพื่อให้เห็นภาพ
ผลลัพธ์.



In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

lead_time = 4
plot_date = start_date + timedelta(hours=int(6 * lead_time))

# โหลดข้อมูลจากร้านค้า zarr ทั้งสองแห่ง
ds0 = xr.open_zarr("outputs/11_hens_0.zarr")
ds1 = xr.open_zarr("outputs/11_hens_1.zarr")

# รวมชุดข้อมูล
ds = xr.concat([ds0, ds1], dim="ensemble")

# คำนวณขนาดความเร็วลม
wind_speed = np.sqrt(ds.u10m**2 + ds.v10m**2)

# รับค่าเฉลี่ยและมาตรฐานของการก้าวครั้งที่ 4 ข้าม ensemble
mean_wind = wind_speed.isel(time=0, lead_time=lead_time).mean(dim="ensemble")
std_wind = wind_speed.isel(time=0, lead_time=lead_time).std(dim="ensemble")

# สร้างรูปที่มีสองแผนย่อย
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(15, 4), subplot_kw={"projection": ccrs.PlateCarree()}
)

# พล็อตหมายถึง
p1 = ax1.contourf(
    mean_wind.coords["lon"],
    mean_wind.coords["lat"],
    mean_wind,
    levels=15,
    transform=ccrs.PlateCarree(),
    cmap="nipy_spectral",
)
ax1.coastlines()
ax1.set_title(f'Mean Wind Speed\n{plot_date.strftime("%Y-%m-%d %H:%M UTC")}')
fig.colorbar(p1, ax=ax1, label="m/s")

# พล็อตส่วนเบี่ยงเบนมาตรฐาน
p2 = ax2.contourf(
    std_wind.coords["lon"],
    std_wind.coords["lat"],
    std_wind,
    levels=15,
    transform=ccrs.PlateCarree(),
    cmap="viridis",
)
ax2.coastlines()
ax2.set_title(
    f'Wind Speed Standard Deviation\n{plot_date.strftime("%Y-%m-%d %H:%M UTC")}'
)
fig.colorbar(p2, ax=ax2, label="m/s")

plt.tight_layout()
# บันทึกรูป
plt.savefig(f"outputs/11_hens_step_{plot_date.strftime('%Y_%m_%d')}.jpg")